# Customer Retention & Revenue Intelligence

## Phase 2: Data Cleaning & Validation

### Objective
Clean and transform the raw customer, usage, and churn datasets based on the data-quality issues identified during profiling, while preserving the original raw files.

### Cleaning Plan
1. Standardize data types.
2. Create an indicator for originally missing Age values.
3. Impute missing Age values using the median age.
4. Remove exact duplicate customer records.
5. Reconcile conflicting SignupDate values using the earliest valid signup date.
6. Validate customer and usage keys.
7. Validate relationships across all three datasets.
8. Export cleaned datasets to the processed data layer.

In [1]:
import pandas as pd

In [2]:
customers = pd.read_csv("../data/raw/customer_info.csv")
usage = pd.read_csv("../data/raw/usage_data.csv")
churn = pd.read_csv("../data/raw/churn_labels.csv")

In [3]:
print("Customers:", customers.shape)
print("Usage:", usage.shape)
print("Churn:", churn.shape)

Customers: (10200, 7)
Usage: (120000, 6)
Churn: (10000, 2)


In [4]:
customers_clean = customers.copy()
usage_clean = usage.copy()
churn_clean = churn.copy()

In [5]:
print("SignupDate before:", customers_clean["SignupDate"].dtype)
print("Month before:", usage_clean["Month"].dtype)

SignupDate before: str
Month before: str


In [6]:
customers_clean["SignupDate"] = pd.to_datetime(
    customers_clean["SignupDate"],
    errors="coerce"
)

usage_clean["Month"] = pd.to_datetime(
    usage_clean["Month"],
    errors="coerce"
)

In [7]:
print("SignupDate after:", customers_clean["SignupDate"].dtype)
print("Month after:", usage_clean["Month"].dtype)

print("Invalid SignupDates:", customers_clean["SignupDate"].isna().sum())
print("Invalid Months:", usage_clean["Month"].isna().sum())

SignupDate after: datetime64[us]
Month after: datetime64[us]
Invalid SignupDates: 0
Invalid Months: 0


In [8]:
customers_clean["AgeMissing"] = customers_clean["Age"].isna().astype(int)

In [9]:
median_age = customers_clean["Age"].median()

customers_clean["Age"] = customers_clean["Age"].fillna(median_age)

In [10]:
print("Median age used:", median_age)
print("Missing Age after cleaning:", customers_clean["Age"].isna().sum())
print("AgeMissing count:", customers_clean["AgeMissing"].sum())

Median age used: 43.0
Missing Age after cleaning: 0
AgeMissing count: 3583


In [11]:
customers_clean[["Age", "AgeMissing"]].head(10)

,Age,AgeMissing
0,43.0,1
1,43.0,1
2,46.0,0
3,43.0,1
4,60.0,0
5,25.0,0
6,38.0,0
7,43.0,1
8,36.0,0
9,43.0,1


In [12]:
customers_clean = customers_clean.sort_values(
    by=["CustomerID", "SignupDate"]
)

In [13]:
customers_clean = customers_clean.drop_duplicates(
    subset="CustomerID",
    keep="first"
)

In [14]:
print("Rows after deduplication:", len(customers_clean))
print("Unique CustomerIDs:", customers_clean["CustomerID"].nunique())
print("Duplicate CustomerIDs:", customers_clean["CustomerID"].duplicated().sum())

Rows after deduplication: 10000
Unique CustomerIDs: 10000
Duplicate CustomerIDs: 0


In [15]:
customers_clean[
    customers_clean["CustomerID"].isin(
        ["CUST_248", "CUST_308", "CUST_333", "CUST_391", "CUST_466"]
    )
][["CustomerID", "SignupDate"]].sort_values("CustomerID")

,CustomerID,SignupDate
10024,CUST_248,2022-06-09
308,CUST_308,2018-01-03
10089,CUST_333,2023-02-11
10150,CUST_391,2023-03-11
466,CUST_466,2021-03-07


In [16]:
print("Customers:", customers_clean["CustomerID"].nunique())
print("Usage customers:", usage_clean["CustomerID"].nunique())
print("Churn customers:", churn_clean["CustomerID"].nunique())

Customers: 10000
Usage customers: 10000
Churn customers: 10000


In [17]:
usage_orphans = set(usage_clean["CustomerID"]) - set(customers_clean["CustomerID"])

print("Usage customers missing from customer table:", len(usage_orphans))

Usage customers missing from customer table: 0


In [18]:
churn_orphans = set(churn_clean["CustomerID"]) - set(customers_clean["CustomerID"])

print("Churn customers missing from customer table:", len(churn_orphans))

Churn customers missing from customer table: 0


In [19]:
print("=== CUSTOMER TABLE ===")
print("Rows:", len(customers_clean))
print("Unique CustomerIDs:", customers_clean["CustomerID"].nunique())
print("Duplicate CustomerIDs:", customers_clean["CustomerID"].duplicated().sum())
print("Missing values:")
print(customers_clean.isnull().sum())

print("\n=== USAGE TABLE ===")
print("Rows:", len(usage_clean))
print("Unique customers:", usage_clean["CustomerID"].nunique())
print(
    "Duplicate CustomerID + Month:",
    usage_clean.duplicated(subset=["CustomerID", "Month"]).sum()
)
print("Missing values:")
print(usage_clean.isnull().sum())

print("\n=== CHURN TABLE ===")
print("Rows:", len(churn_clean))
print("Unique CustomerIDs:", churn_clean["CustomerID"].nunique())
print("Duplicate CustomerIDs:", churn_clean["CustomerID"].duplicated().sum())
print("Missing values:")
print(churn_clean.isnull().sum())

=== CUSTOMER TABLE ===
Rows: 10000
Unique CustomerIDs: 10000
Duplicate CustomerIDs: 0
Missing values:
CustomerID        0
Age               0
Gender            0
Region            0
ContractType      0
MonthlyCharges    0
SignupDate        0
AgeMissing        0
dtype: int64

=== USAGE TABLE ===
Rows: 120000
Unique customers: 10000
Duplicate CustomerID + Month: 0
Missing values:
CustomerID     0
Month          0
CallMinutes    0
DataUsageGB    0
SMSCount       0
Complaints     0
dtype: int64

=== CHURN TABLE ===
Rows: 10000
Unique CustomerIDs: 10000
Duplicate CustomerIDs: 0
Missing values:
CustomerID    0
Churn         0
dtype: int64


## Business Rule Validation

Validate cleaned data against expected business rules and logical value ranges before exporting the processed datasets.

In [20]:
print("Age range:",
      customers_clean["Age"].min(),
      "to",
      customers_clean["Age"].max())

print("Monthly Charges range:",
      customers_clean["MonthlyCharges"].min(),
      "to",
      customers_clean["MonthlyCharges"].max())

print("Gender values:",
      customers_clean["Gender"].unique())

print("Region values:",
      customers_clean["Region"].unique())

print("Contract types:",
      customers_clean["ContractType"].unique())

print("Churn values:",
      churn_clean["Churn"].unique())

Age range: 18.0 to 69.0
Monthly Charges range: -6.72 to 390.3
Gender values: <StringArray>
['Male', 'Female']
Length: 2, dtype: str
Region values: <StringArray>
['Urban', 'Suburban', 'Rural']
Length: 3, dtype: str
Contract types: <StringArray>
['Yearly', 'Prepaid', 'Monthly']
Length: 3, dtype: str
Churn values: [0 1]


In [21]:
print("Call Minutes:",
      usage_clean["CallMinutes"].min(),
      "to",
      usage_clean["CallMinutes"].max())

print("Data Usage GB:",
      usage_clean["DataUsageGB"].min(),
      "to",
      usage_clean["DataUsageGB"].max())

print("SMS Count:",
      usage_clean["SMSCount"].min(),
      "to",
      usage_clean["SMSCount"].max())

print("Complaints:",
      usage_clean["Complaints"].min(),
      "to",
      usage_clean["Complaints"].max())

Call Minutes: 34.14 to 914.16
Data Usage GB: 0.0 to 22.73
SMS Count: 0 to 49
Complaints: 0 to 6


A negative monthly charge of -$6.72

In [22]:
invalid_charges = customers_clean[
    customers_clean["MonthlyCharges"] <= 0
]

invalid_charges

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,AgeMissing
4671,CUST_4671,56.0,Male,Rural,Prepaid,-3.89,2020-11-12,0
5575,CUST_5575,39.0,Female,Suburban,Monthly,-0.46,2023-02-01,0
6820,CUST_6820,60.0,Male,Urban,Yearly,-3.15,2018-02-09,0
8567,CUST_8567,36.0,Male,Rural,Prepaid,-6.72,2018-04-17,0
9403,CUST_9403,19.0,Male,Rural,Monthly,-4.55,2023-05-18,0


In [23]:
len(invalid_charges)

5

In [24]:
customers_clean["MonthlyCharges"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    10000.000000
mean        51.000240
std         21.543385
min         -6.720000
1%          14.269800
5%          24.968000
25%         39.797500
50%         50.015000
75%         60.500000
95%         75.710500
99%         89.420700
max        390.300000
Name: MonthlyCharges, dtype: float64

In [25]:
high_charges = customers_clean[
    customers_clean["MonthlyCharges"] > 100
].sort_values("MonthlyCharges", ascending=False)

high_charges

,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,AgeMissing
583,CUST_583,43.0,Female,Suburban,Prepaid,390.30,2018-01-07,1
9404,CUST_9404,43.0,Male,Urban,Monthly,369.55,2019-11-19,1
3666,CUST_3666,21.0,Male,Urban,Prepaid,367.65,2021-09-09,0
5772,CUST_5772,35.0,Male,Urban,Monthly,356.00,2023-06-09,0
988,CUST_988,31.0,Male,Rural,Prepaid,353.35,2022-12-25,0
8612,CUST_8612,22.0,Male,Urban,Prepaid,338.10,2018-01-24,0
8715,CUST_8715,43.0,Male,Suburban,Prepaid,329.95,2023-05-30,1
6545,CUST_6545,62.0,Male,Urban,Yearly,321.45,2023-01-10,0
2155,CUST_2155,48.0,Female,Rural,Monthly,318.50,2021-05-11,0
5224,CUST_5224,43.0,Female,Rural,Yearly,316.80,2021-02-09,1


In [26]:
len(high_charges)

57

In [27]:
print("Above $100:", (customers_clean["MonthlyCharges"] > 100).sum())
print("Above $150:", (customers_clean["MonthlyCharges"] > 150).sum())
print("Above $200:", (customers_clean["MonthlyCharges"] > 200).sum())
print("Above $300:", (customers_clean["MonthlyCharges"] > 300).sum())

Above $100: 57
Above $150: 48
Above $200: 38
Above $300: 15


In [28]:
customers_clean.groupby("ContractType")["MonthlyCharges"].agg(
    Count="count",
    Median="median",
    Mean="mean"
)

,Count,Median,Mean
ContractType,,,
Monthly,3953,49.70,50.844010
Prepaid,4041,50.08,51.091391
Yearly,2006,50.39,51.124487


In [29]:
customers_clean.groupby(
    ["ContractType", "Region"]
)["MonthlyCharges"].median()

ContractType  Region  
Monthly       Rural       49.57
              Suburban    49.93
              Urban       49.69
Prepaid       Rural       49.77
              Suburban    50.31
              Urban       50.11
Yearly        Rural       50.59
              Suburban    49.64
              Urban       50.99
Name: MonthlyCharges, dtype: float64

### Monthly Charges Review

Business-rule validation identified five customers with negative `MonthlyCharges`.

Because the dataset does not specify whether negative charges represent data errors, promotional credits, refunds, billing adjustments, or other legitimate business activity, the original values were retained rather than imputed or removed.

A `ChargeReviewFlag` was created to identify these records for further business review.

High positive charges were also retained because statistical outliers do not necessarily represent invalid data.

In [30]:
customers_clean["ChargeReviewFlag"] = (
    customers_clean["MonthlyCharges"] < 0
).astype(int)

In [31]:
print(customers_clean["ChargeReviewFlag"].value_counts())

customers_clean[
    customers_clean["ChargeReviewFlag"] == 1
][
    ["CustomerID", "ContractType", "Region", "MonthlyCharges", "ChargeReviewFlag"]
]

ChargeReviewFlag
0    9995
1       5
Name: count, dtype: int64


,CustomerID,ContractType,Region,MonthlyCharges,ChargeReviewFlag
4671,CUST_4671,Prepaid,Rural,-3.89,1
5575,CUST_5575,Monthly,Suburban,-0.46,1
6820,CUST_6820,Yearly,Urban,-3.15,1
8567,CUST_8567,Prepaid,Rural,-6.72,1
9403,CUST_9403,Monthly,Rural,-4.55,1


## Final Data Quality Validation

Perform final structural, completeness, key-integrity, and business-rule checks before exporting the processed datasets.

In [32]:
print("=== CUSTOMERS ===")
print("Rows:", len(customers_clean))
print("Unique CustomerIDs:", customers_clean["CustomerID"].nunique())
print("Duplicate CustomerIDs:", customers_clean["CustomerID"].duplicated().sum())
print("Missing values:", customers_clean.isna().sum().sum())
print("Age range:", customers_clean["Age"].min(), "to", customers_clean["Age"].max())
print("Age values imputed:", customers_clean["AgeMissing"].sum())
print("Charges flagged for review:", customers_clean["ChargeReviewFlag"].sum())

print("\n=== USAGE ===")
print("Rows:", len(usage_clean))
print("Unique customers:", usage_clean["CustomerID"].nunique())
print("Duplicate CustomerID + Month:",
      usage_clean.duplicated(subset=["CustomerID", "Month"]).sum())
print("Missing values:", usage_clean.isna().sum().sum())

print("\n=== CHURN ===")
print("Rows:", len(churn_clean))
print("Unique CustomerIDs:", churn_clean["CustomerID"].nunique())
print("Duplicate CustomerIDs:", churn_clean["CustomerID"].duplicated().sum())
print("Missing values:", churn_clean.isna().sum().sum())
print("Valid Churn values:", sorted(churn_clean["Churn"].unique()))

print("\n=== REFERENTIAL INTEGRITY ===")
print(
    "Usage orphan customers:",
    len(set(usage_clean["CustomerID"]) - set(customers_clean["CustomerID"]))
)
print(
    "Churn orphan customers:",
    len(set(churn_clean["CustomerID"]) - set(customers_clean["CustomerID"]))
)

=== CUSTOMERS ===
Rows: 10000
Unique CustomerIDs: 10000
Duplicate CustomerIDs: 0
Missing values: 0
Age range: 18.0 to 69.0
Age values imputed: 3500
Charges flagged for review: 5

=== USAGE ===
Rows: 120000
Unique customers: 10000
Duplicate CustomerID + Month: 0
Missing values: 0

=== CHURN ===
Rows: 10000
Unique CustomerIDs: 10000
Duplicate CustomerIDs: 0
Missing values: 0
Valid Churn values: [np.int64(0), np.int64(1)]

=== REFERENTIAL INTEGRITY ===
Usage orphan customers: 0
Churn orphan customers: 0


Earlier, before deduplication, we had 3,583 missing Age values. After reducing 10,200 rows to 10,000 unique customers,

In [33]:
raw_missing_age_customers = customers.loc[
    customers["Age"].isna(),
    "CustomerID"
].nunique()

print("Unique customers with missing Age in raw data:",
      raw_missing_age_customers)

print("AgeMissing after deduplication:",
      customers_clean["AgeMissing"].sum())

Unique customers with missing Age in raw data: 3500
AgeMissing after deduplication: 3500


## Export Processed Data

After completing data cleaning and validation, the analytics-ready datasets are exported to the processed data layer while preserving the original raw source files.

In [35]:
customers_clean.to_csv(
    "../data/processed/customers_clean.csv",
    index=False
)

usage_clean.to_csv(
    "../data/processed/usage_clean.csv",
    index=False
)

churn_clean.to_csv(
    "../data/processed/churn_clean.csv",
    index=False
)

print("Processed datasets exported successfully.")

Processed datasets exported successfully.


In [36]:
customers_test = pd.read_csv("../data/processed/customers_clean.csv")
usage_test = pd.read_csv("../data/processed/usage_clean.csv")
churn_test = pd.read_csv("../data/processed/churn_clean.csv")

print("Customers:", customers_test.shape)
print("Usage:", usage_test.shape)
print("Churn:", churn_test.shape)

Customers: (10000, 9)
Usage: (120000, 6)
Churn: (10000, 2)
